<a href="https://colab.research.google.com/github/MsTop17/RAG_Projects_basic_to_Advanced/blob/main/Rag_project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hello")

hello


In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini connection successful!")

Gemini connection successful!


In [ ]:
response = client.models.generate_content(
    model="gemini-2.0-flash",  # or "gemini-1.5-flash"
    contents="Explain what RAG is in two simple sentences."
)
print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 20.949449036s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go

In [ ]:
for model in client.models.list():
    print(model.name)

In [ ]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain what RAG is in two simple sentences."
)

print(response.text)

RAG is a technique that allows an AI to look up relevant information from a specific database or external source before answering a question. By combining this retrieved data with its own knowledge, the AI can provide more accurate, up-to-date, and factual responses.


In [ ]:
documents = {
    "torpor.txt": """
Torpor is a physiological state in which an animal temporarily
reduces its metabolic activity. During torpor, physiological
processes can slow down and energy expenditure can decrease.
Torpor occurs naturally in several animal species and is studied
as a possible concept for future long-duration spaceflight.
""",

    "neuroscience.txt": """
Neuroscience is the scientific study of the nervous system.
The brain contains billions of neurons that communicate with
each other. Researchers study brain activity using different
methods, including electrical signals such as EEG.
""",

    "spaceflight.txt": """
Long-duration spaceflight presents several challenges, including
microgravity, radiation, isolation and limited resources.
Researchers investigate different approaches that could help
humans tolerate future long-duration missions. Torpor is one
concept being investigated for possible future spaceflight.
"""
}

print("Documents loaded:")
for name in documents:
    print("-", name)

Documents loaded:
- torpor.txt
- neuroscience.txt
- spaceflight.txt


In [ ]:
def chunk_text(text, chunk_size=200):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks

In [ ]:
all_chunks = []
for filename,text in documents.items():
  chunks = chunk_text(text)
  for i, chunk in enumerate(chunks):
    all_chunks.append({
        "source":filename,
        "chunk_id": i,
        "text":chunk
    })
print("Total Chunks :" , len(all_chunks))

Total Chunks : 6


In [ ]:
for chunk in all_chunks:
    print("SOURCE:", chunk["source"])
    print("CHUNK:", chunk["chunk_id"])
    print(chunk["text"])
    print("-" * 60)

SOURCE: torpor.txt
CHUNK: 0

Torpor is a physiological state in which an animal temporarily
reduces its metabolic activity. During torpor, physiological
processes can slow down and energy expenditure can decrease.
Torpor occurs 
------------------------------------------------------------
SOURCE: torpor.txt
CHUNK: 1
naturally in several animal species and is studied
as a possible concept for future long-duration spaceflight.

------------------------------------------------------------
SOURCE: neuroscience.txt
CHUNK: 0

Neuroscience is the scientific study of the nervous system.
The brain contains billions of neurons that communicate with
each other. Researchers study brain activity using different
methods, includin
------------------------------------------------------------
SOURCE: neuroscience.txt
CHUNK: 1
g electrical signals such as EEG.

------------------------------------------------------------
SOURCE: spaceflight.txt
CHUNK: 0

Long-duration spaceflight presents several challe

In [ ]:
def get_embedding(text):

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )

    return result.embeddings[0].values

In [ ]:
vector = get_embedding("What is torpor?")

print("Embedding created!")
print("Number of values:", len(vector))
print("First 5 values:", vector[:5])

Embedding created!
Number of values: 3072
First 5 values: [0.003432764, 0.032310214, 0.0026477578, -0.0705197, 0.011016612]


In [ ]:
for chunk in all_chunks:
    chunk["embedding"] = get_embedding(
        chunk["text"]
    )
print("All chunks embedded!")

All chunks embedded!


In [ ]:
import numpy as np

In [ ]:
def cosine_similarity(a,b):
  a = np.array(a)
  b = np.array(b)
  return np.dot(a,b)/(
      np.linalg.norm(a)*np.linalg.norm(b)
  )

In [ ]:
def search_documents(query, top_k=2):

    query_embedding = get_embedding(query)

    results = []

    for chunk in all_chunks:

        score = cosine_similarity(
            query_embedding,
            chunk["embedding"]
        )

        results.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "score": score
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]

In [ ]:
results = search_documents("What is torpor?", top_k=2)

for r in results:
    print(r["source"], "|", r["chunk_id"], "|", round(r["score"], 3))

torpor.txt | 0 | 0.839
spaceflight.txt | 1 | 0.768


In [ ]:
results = search_documents("What is neuroscience?", top_k=2)

for r in results:
    print(r["source"], "|", r["chunk_id"], "|", round(r["score"], 3))

neuroscience.txt | 0 | 0.778
neuroscience.txt | 1 | 0.602


In [ ]:
results = search_documents(
    "What is neuroscience?",
    top_k=2
)

for r in results:
    print("\nSource:", r["source"])
    print("Chunk:", r["chunk_id"])
    print("Score:", round(r["score"], 4))
    print("Text:", r["text"])


Source: neuroscience.txt
Chunk: 0
Score: 0.778
Text: 
Neuroscience is the scientific study of the nervous system.
The brain contains billions of neurons that communicate with
each other. Researchers study brain activity using different
methods, includin

Source: neuroscience.txt
Chunk: 1
Score: 0.6022
Text: g electrical signals such as EEG.



In [ ]:
results = search_documents(
    "What is neuroscience?",
    top_k=2
)

print("Number of results:", len(results))

for i, result in enumerate(results):
    print("\nRESULT", i + 1)
    print("Source:", result["source"])
    print("Chunk:", result["chunk_id"])
    print("Score:", round(result["score"], 3))
    print("Text:", result["text"])

Number of results: 2

RESULT 1
Source: spaceflight.txt
Chunk: 1
Score: 0.522
Text:  future long-duration missions. Torpor is one
concept being investigated for possible future spaceflight.


RESULT 2
Source: spaceflight.txt
Chunk: 1
Score: 0.522
Text:  future long-duration missions. Torpor is one
concept being investigated for possible future spaceflight.



In [ ]:
print("Total chunks:", len(all_chunks))

for i, chunk in enumerate(all_chunks):
    print(
        i,
        "|",
        chunk["source"],
        "| chunk", chunk["chunk_id"],
        "|",
        chunk["text"][:80].replace("\n", " ")
    )

Total chunks: 6
0 | torpor.txt | chunk 0 |  Torpor is a physiological state in which an animal temporarily reduces its meta
1 | torpor.txt | chunk 1 | naturally in several animal species and is studied as a possible concept for fut
2 | neuroscience.txt | chunk 0 |  Neuroscience is the scientific study of the nervous system. The brain contains 
3 | neuroscience.txt | chunk 1 | g electrical signals such as EEG. 
4 | spaceflight.txt | chunk 0 |  Long-duration spaceflight presents several challenges, including microgravity, 
5 | spaceflight.txt | chunk 1 |  future long-duration missions. Torpor is one concept being investigated for pos


In [ ]:
for chunk in all_chunks:
    print(
        chunk["source"],
        "chunk", chunk["chunk_id"],
        "embedding length:", len(chunk["embedding"]),
        "first 3 values:", chunk["embedding"][:3]
    )

torpor.txt chunk 0 embedding length: 3072 first 3 values: [0.028682873, 0.025253367, 0.0066021252]
torpor.txt chunk 1 embedding length: 3072 first 3 values: [0.0025709793, 0.012780032, 0.0122949565]
neuroscience.txt chunk 0 embedding length: 3072 first 3 values: [-0.009154552, 0.00023169213, -0.0011021044]
neuroscience.txt chunk 1 embedding length: 3072 first 3 values: [0.024972824, 0.008996856, 0.009339998]
spaceflight.txt chunk 0 embedding length: 3072 first 3 values: [0.019763999, -0.0014399772, 0.02936349]
spaceflight.txt chunk 1 embedding length: 3072 first 3 values: [-0.005327116, 0.0227768, 0.014297648]


In [ ]:
query_embedding = get_embedding("What is neuroscience?")

for chunk in all_chunks:

    score = cosine_similarity(
        query_embedding,
        chunk["embedding"]
    )

    print(
        chunk["source"],
        "| chunk", chunk["chunk_id"],
        "| score:", round(score, 4)
    )

torpor.txt | chunk 0 | score: 0.4851
torpor.txt | chunk 1 | score: 0.501
neuroscience.txt | chunk 0 | score: 0.778
neuroscience.txt | chunk 1 | score: 0.6022
spaceflight.txt | chunk 0 | score: 0.5557
spaceflight.txt | chunk 1 | score: 0.5222


In [ ]:
results = search_documents(
    "What challenges exist during long-duration spaceflight?",
    top_k=2
)

for r in results:
    print("\nSource:", r["source"])
    print("Chunk:", r["chunk_id"])
    print("Score:", round(r["score"], 4))
    print("Text:", r["text"])


Source: spaceflight.txt
Chunk: 0
Score: 0.8473
Text: 
Long-duration spaceflight presents several challenges, including
microgravity, radiation, isolation and limited resources.
Researchers investigate different approaches that could help
humans tolerate

Source: spaceflight.txt
Chunk: 1
Score: 0.7391
Text:  future long-duration missions. Torpor is one
concept being investigated for possible future spaceflight.



In [ ]:
def build_prompt(question, retrieved_chunks):

    context = "\n\n".join(
        [
            f"Source: {chunk['source']}\n{chunk['text']}"
            for chunk in retrieved_chunks
        ]
    )

    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the information provided
in the context below.

If the answer cannot be found in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [ ]:
def generate_answer(question, top_k=2):

    # Retrieve relevant chunks
    retrieved_chunks = search_documents(
        question,
        top_k=top_k
    )

    # Build the prompt
    prompt = build_prompt(
        question,
        retrieved_chunks
    )

    # Ask Gemini
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text, retrieved_chunks

In [ ]:
answer, sources = generate_answer(
    "What is torpor?"
)

print("ANSWER:")
print(answer)

print("\nSOURCES USED:")
for source in sources:
    print(
        "-",
        source["source"],
        "chunk",
        source["chunk_id"]
    )

ANSWER:
Torpor is a physiological state in which an animal temporarily reduces its metabolic activity. During this state, physiological processes can slow down, energy expenditure can decrease, and it is a concept being investigated for possible future spaceflight.

SOURCES USED:
- torpor.txt chunk 0
- spaceflight.txt chunk 1


                 YOUR DOCUMENTS
                       ↓
                    CHUNKING
                       ↓
                   EMBEDDINGS
                       ↓
                  VECTOR SPACE
                       ↓
Question → EMBEDDING → SIMILARITY SEARCH
                       ↓
                 RELEVANT CHUNKS
                       ↓
                PROMPT + CONTEXT
                       ↓
                    GEMINI
                       ↓
                    ANSWER

In [ ]:
answer, sources = generate_answer(
    "What is the capital of France?"
)

print("ANSWER:")
print(answer)

ANSWER:
I don't know based on the provided documents.


In [ ]:
answer, sources = generate_answer(
    "What challenges are associated with long-duration spaceflight?"

)

print(answer)

print("\nSources:")
for s in sources:
    print("-", s["source"])

The challenges associated with long-duration spaceflight include microgravity, radiation, isolation, and limited resources.

Sources:
- spaceflight.txt
- spaceflight.txt


📁 Basic-RAG
│
├── 📄 torpor.txt
├── 📄 neuroscience.txt
├── 📄 spaceflight.txt
│
├── Chunking
├── Embeddings
├── Similarity Search
├── Retrieval
├── Prompt Construction
└── Gemini Generation